# Kaggle Smoke Test - Plant Classification

Notebook nay dung de kiem tra pipeline nhanh nhat co the truoc khi train full.

Muc tieu:
- load dataset thanh cong
- train duoc 1 epoch
- tao metrics co ban
- save model weights
- save confusion matrix

Mac dinh notebook nay chi dung mot tap con nho cua dataset de chay nhanh tren Kaggle.

In [ ]:
from pathlib import Path

# Kaggle dataset path da xac nhan
DATASET_DIR = Path('/kaggle/input/datasets/thngbuduc/plant-classification/dataset_plant_classification')
OUTPUT_DIR = Path('/kaggle/working/smoke_test_outputs')

IMAGE_SIZE = 128
MAX_CLASSES = 8
MAX_IMAGES_PER_CLASS = 80
TRAIN_RATIO = 0.8
BATCH_SIZE = 32
EPOCHS = 1
NUM_WORKERS = 2
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR

In [ ]:
import json
import random
import time
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import seaborn as sns

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png'}

class_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])[:MAX_CLASSES]
class_names = [p.name for p in class_dirs]

samples = []
per_class_counts = {}

for class_idx, class_dir in enumerate(class_dirs):
    images = sorted([
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in VALID_EXTS
    ])[:MAX_IMAGES_PER_CLASS]
    per_class_counts[class_dir.name] = len(images)
    for img_path in images:
        samples.append((img_path, class_idx))

print('Selected classes:', class_names)
print('Images per class:', per_class_counts)
print('Total selected images:', len(samples))

assert len(class_names) >= 2, 'Can it nhat 2 class de smoke test.'
assert len(samples) > 0, 'Khong tim thay anh nao trong DATASET_DIR.'

In [ ]:
by_class = {}
for img_path, label in samples:
    by_class.setdefault(label, []).append((img_path, label))

train_samples = []
val_samples = []

for label, items in by_class.items():
    random.shuffle(items)
    split_idx = max(1, int(len(items) * TRAIN_RATIO))
    split_idx = min(split_idx, len(items) - 1) if len(items) > 1 else 1
    train_samples.extend(items[:split_idx])
    val_samples.extend(items[split_idx:])

print('Train samples:', len(train_samples))
print('Val samples:', len(val_samples))

In [ ]:
class ImageListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label = self.samples[index]
        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageListDataset(train_samples, transform=train_transform)
val_dataset = ImageListDataset(val_samples, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_loader), len(val_loader)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = TinyCNN(num_classes=len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_labels, all_preds

In [ ]:
history = []
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, val_acc, val_labels, val_preds = run_epoch(model, val_loader, optimizer=None)

    epoch_result = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    }
    history.append(epoch_result)
    print(epoch_result)

elapsed = time.time() - start_time
print(f'Total runtime: {elapsed:.2f} seconds')

In [ ]:
cm = confusion_matrix(val_labels, val_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Smoke Test Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'smoke_confusion_matrix.png', dpi=200)
plt.show()

In [ ]:
torch.save(model.state_dict(), OUTPUT_DIR / 'tiny_cnn_smoke_test.pth')

summary = {
    'dataset_dir': str(DATASET_DIR),
    'device': str(DEVICE),
    'image_size': IMAGE_SIZE,
    'max_classes': MAX_CLASSES,
    'max_images_per_class': MAX_IMAGES_PER_CLASS,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'class_names': class_names,
    'per_class_counts': per_class_counts,
    'history': history,
    'runtime_seconds': elapsed,
}

with open(OUTPUT_DIR / 'smoke_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

with open(OUTPUT_DIR / 'class_names.txt', 'w', encoding='utf-8') as f:
    for name in class_names:
        f.write(name + '\n')

print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)